# ŞABLON — deneme 2

**Kullanımı:** bu defteri **kopyala**, adını modelin adı yap
(`model_a.ipynb`), ve **yalnız aşağıdaki tek satırı** değiştir.

Başka hiçbir yeri elle düzenleme. Eskiden HÜCRE 0 önceki deneyin ayarlarını
taşıyordu ve düzeltmeyi unutan kişi sessizce önceki deneyi başlatıyordu.

**Kuyruk / sürücü yok.** Bir defter = bir model. Sırayla koşmak istiyorsan
iki defter aç.

In [ ]:
MODEL = None               # <-- DEGISTIRILECEK TEK SATIR, orn. "model_a"

assert MODEL, ("MODEL bos. Bu SABLON -- kopyala, adini modelin adi yap "
               "(model_a.ipynb) ve bu satiri doldur.")
import time
DEPO  = "https://github.com/sekerahmet/sekerai.git"
KOD   = "/content/kod"
EV    = f"/content/drive/MyDrive/{MODEL}"     # TEPE KLASOR = MODELIN ADI
DAMGA = time.strftime("%Y%m%d_%H%M%S")
LOG   = f"{EV}/log/kos_{DAMGA}.txt"
# LOG'da ZAMAN DAMGASI var, cunku eskiden tek `log.txt` vardi ve her kosu
# onu "w" ile aciyordu: TOHUMLAR=[1,2] ile ikinci kez kosunca t0'in logu
# SILINIYORDU. Oysa kural "t0 klasorune dokunulmaz" idi.
# Log TOHUMUN degil KOSUNUN: bir kosu birden cok tohum surebilir.
print(MODEL, "->", EV)
print("log:", LOG)

## 1 — GPU

CPU'ya düşerse 45 dakikalık iş **36 saat** olur (ölçüldü: CPU'da adım başına
1,64 sn). Burada durmak, 36 saat sonra fark etmekten iyidir.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir > T4"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos -- baska bir sey mi kosuyor?"

## 2 — Drive

Çıktı doğrudan Drive'a yazılır. `/content` runtime ölünce silinir, Drive silinmez.

`drive.mount` çalışmazsa `/content/drive/MyDrive/...` **sihirli bir yol
değildir** — sıradan bir klasördür ve `os.makedirs` onu geçici diskte
sessizce açar. 45 dakika koşar, biter, runtime ölür, her şey silinir.
`ismount` bunu ayırır.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), \
    "drive.mount CALISMADI -- /content/drive gercek bir baglanti degil."
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  var olan tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

## 3 — Kod

**Colab'da yama yok.** Depo silinip yeniden klonlanır, üzerine hiçbir şey
yazılmaz. Tek kaynak: GitHub.

Yereli düzeltip itmeyi unuttuysan burada görürsün — klon eski commit'i getirir.

In [ ]:
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())

# Her modelin AILE klasoru var: deneme2/model_a/{model_a.py, pencere_a.py,
# model_a.ipynb}. Klasor adini isimden TURETMIYORUZ, dosyayi ARIYORUZ --
# model_a1 gibi varyasyonlar da ayni aile klasorunde durur.
_aday = glob.glob(f"{KOD}/deneme2/*/{MODEL}.py")
assert len(_aday) == 1, f"{MODEL}.py tam bir kez bulunmali, bulunan: {_aday}"
AILE = os.path.dirname(_aday[0])
_pen = glob.glob(f"{AILE}/pencere_*.py")
assert len(_pen) == 1, f"ailede tam bir pencere_*.py olmali: {_pen}"
PENCERE = _pen[0]
print("aile:", AILE, "| olcum:", os.path.basename(PENCERE))

sys.path.insert(0, AILE)
M = importlib.import_module(MODEL)
assert M.AYAR.ad == MODEL, f"AYAR.ad {M.AYAR.ad!r} != dosya adi {MODEL!r}"
print("ayar:", M.AYAR)

## 4 — Başlat

**Önce tek tohum** (`t0`). Sonuç olumluysa yeter; olumsuzsa `TOHUMLAR`'ı
`[1, 2]` yapıp tekrar koşarsın — `t0` klasörüne dokunulmaz.

Çıktı Drive'da, **tepe klasör modelin adı**:

```
MyDrive/<model>/
    OKU.md        klasörü anlatır, her koşuda yenilenir
    log/          kos_<zaman>.txt
    t0/           ayar_ · kosu_ · egri_ · pencere_ json'ları
        snap/     snap_<model>_t0_00020000.pt
```

Dosya adları da tohumu taşır — klasörden çıkarmak gerekmiyor.

**Ayrı süreç.** Çekirdek serbest kalır: ilerlemeye bakabilir, durdurabilirsin.

Koşan kod `deneme2/kos.py` — **depoda**, defterin içinde metin olarak
kurulmuyor. Yani başlatan kod da koşan kodla aynı commit'te.

In [ ]:
TOHUMLAR = [0]             # ONCE TEK TOHUM.
# Sonuc OLUMLU cikarsa (ent yukseldiyse) tek tohum YETER.
# OLUMSUZ cikarsa bu satiri [1, 2] yapip tekrar kos -- t0 klasoru
# DOKUNULMAZ, yeni kosular t1/ ve t2/'ye yazar. Kod degismez.
#
# Neden onemli: grokking tohuma bagli (2603.25009'da "only 1 of 3 seeds
# grokked" gibi sonuclar var). Tek tohumda ent yukselmezse
# "konfigurasyon yanlis" ile "bu baslangic sanssiz" AYRILAMAZ.
# VERI tohumu AYRI (ayar.veri_tohum=0) -- butun tohumlar AYNI veriyi gorur.

# IKINCI SURECI ENGELLE: bu hucre iki kez calistirilirsa iki egitim AYNI
# klasore yazar ve birbirinin anlik goruntusunu ezer. Hicbiri hata vermez.
if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once 'Durdurmak' hucresi.")

p = subprocess.Popen(
    [sys.executable, "-u", f"{KOD}/deneme2/kos.py",
     "--model", MODEL, "--ev", EV, "--commit", COMMIT,
     "--tohum", *[str(t) for t in TOHUMLAR]],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kos.py REDDEDER (uzerine yazmaz).")

## 5 — İlerleme

Bu hücre **hesap yapmaz, GPU kullanmaz** — sadece dosya okur. Koşu
kilitliyken bile çalışır. İstediğin kadar tekrar çalıştır.

In [ ]:
_d = p.poll()
print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis kodu {_d})", "| PID", p.pid)
print("-" * 78)
print(open(LOG).read()[-4000:])

## 6 — Rapor

**GPU kullanmaz, koşu sürerken çalışır.** Eğri ve künye dosyalarını Drive'dan
okur; ikisi de **atomik** yazılıyor, yani yarım dosya okunmaz.

> Bu bir **ön okuma**. Birincil okuma `pencere_a.py` — o, N anlık görüntünün
> **ağırlık ortalamasını** alıp tek model ölçer. Eğri değerleriyle pencere
> değerleri **aynı şey değildir**; arşivde ikisi ~2× farklı çıktı.

In [ ]:
import json, glob, os

SUT = (("adim", "adim"), ("kayip", "kayip"), ("one", "one"), ("seen", "seen"),
       ("comp", "comp"), ("ent", "ent"), ("ent_kisayol", "ent_ksy"),
       ("ent_yok_kisayol", "yok_ksy"))

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{os.path.basename(kl)}: egri YOK"); continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    print("=" * 84)
    print(f"{os.path.basename(kl)}   {k.get('ad','?')}   durum {k.get('durum','?')}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    v = k.get("veri", {})
    if v:
        print(f"   olgu {v['olgu']}  egitim2 {v['egitim2']}  ENT {v['ent']}  "
              f"phi {v['phi']} (wang {v['wang_phi']})  "
              f"parametre {k.get('parametre',0):,}  iz {k.get('olcme_izi','?')}")
    print("   " + "".join(f"{b:>10}" for _, b in SUT) + f"{'dk':>6}")
    for r in e:
        hcr = []
        for c, _ in SUT:
            x = r.get(c)
            hcr.append(f"{x:>10d}" if c == "adim" else
                       (f"{'---':>10}" if x is None else f"{x:>10.4f}"))
        print("   " + "".join(hcr) + f"{r['sn']/60:>6.0f}")

    s = e[-1]
    print("   " + "-" * 81)
    for ad, kural, g in (
            ("SAGLIK-1HOP",  "one >= 0.98",  s.get("one", 0) >= 0.98),
            ("SAGLIK-EZBER", "seen >= 0.95", s.get("seen", 0) >= 0.95),
            ("OLGUNLUK",     "comp >= 0.50", s.get("comp", 0) >= 0.50),
            ("BIRIM TESTI",  "ent_yok_kisayol == 0",
             abs(s.get("ent_yok_kisayol", 1)) < 1e-9)):
        print(f"   {'GECTI ' if g else '!! KALDI'}  {ad:<14} {kural}")
    if len(e) >= 2:
        print(f"   ent son iki olcumde {e[-1].get('ent',0)-e[-2].get('ent',0):+.4f}"
              "   (BUTCE hukmu pencere_a'da verilir, burada DEGIL)")

## 7 — Eğitim koşarken ne çalıştırabilirim?

```
GPU KULLANMAZ -- istedigin kadar, istedigin zaman
    5 Ilerleme (log okur)        6 Rapor (json okur)
    Anlik goruntu ve egri ATOMIK yaziliyor (.tmp -> replace):
    okuyucu ya ESKI ya YENI dosyayi gorur, ARASINI asla.
    Olculdu: atomik olmadan yarim .pt torch.load'i patlatiyor ve
    pencere_a'nin glob'una giriyordu.

GPU ISTER -- ayni runtime'da calisir ama GPU'yu PAYLASIR
    8 pencere_a.  Model 3,4M parametre; T4'un 15 GB'inda yer sorun degil,
    ama egitim YAVASLAR. Acelen yoksa kosu bitince calistir.

IKINCI DEFTER = IKINCI RUNTIME = IKINCI GPU
    Colab'da yeni bir defter acmak ayni GPU'yu paylasmaz, AYRI bir runtime
    verir. Orada sadece 2 (Drive) + 3 (Kod) + 8 (pencere_a) hucrelerini
    calistir -- 4 (Baslat) hucresine DOKUNMA. Egitim hic etkilenmez.
    Bedeli: ayni anda iki Colab oturumu (kota).
```

In [ ]:
# BIRINCIL OKUMA. GPU kullanir (yukaridaki nota bak).
# PENCERE hucre 3'te ailenin icinden bulundu -- yolu elle yazmiyoruz.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

## 8 — Durdurmak

`p.kill()` — ya da runtime'ı kapat. Anlık görüntüler Drive'da kalır ve
`durum` alanı `HATA`/`KOSUYOR`da kalır, yani rapor koşunun **bitmediğini**
söyler.

**Sürdürme yok.** Koşu kesilirse baştan başlar (~45 dk). 20.000 adım için
bu kabul edilebilir; olmadığı gün sürdürme eklenir.

Yarım klasöre tekrar koşmak istersen `kos.py` **reddeder**; bilerek ezmek
için `--ustune`.

In [ ]:
# p.kill()